# Task 3.2 — Failure Mode
**Paper:** One-Sided Support Vector Regression for Multiclass Cost-Sensitive Classification (Tu & Lin, ICML 2010)  
**Student:** Vaishnav Verma (230160)

## Failure Scenario: Uniform Cost Matrix

I construct a scenario where OSSVR's core advantage — preserving the ordinal cost structure through regression — is **neutralised**. When the cost matrix is uniform (all off-diagonal costs are equal), the regret vector becomes a binary indicator: 0 for the correct class and a constant c for every wrong class. The regression targets carry no more information than a standard one-hot label, and OSSVR is solving a harder problem (regression) to achieve the same outcome that a simple classifier can achieve directly.

**Why I expect the method to struggle:** OSSVR is designed to exploit differences in cost magnitudes across classes. With uniform costs, there are no magnitude differences — all misclassifications are equally bad. The one-sided loss wastes model capacity by allowing free over-estimation of a constant target, and the regression framework adds complexity without benefit. A standard SVM, which is natively designed for this equal-cost setting, should perform at least as well with less computational overhead.

**Connection to Assumption 1 (Task 1.2):** This failure mode directly connects to my first assumption — that the cost matrix is known and *meaningful* (non-uniform). OSSVR's method construction assumes the cost structure provides useful signal for the regression. When costs are uniform, this assumption is technically satisfied but vacuously so — the cost information is trivial.

In [1]:
# === Configuration & Seeds ===
import numpy as np
import random
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

TEST_SIZE = 0.3
C_REG = 100.0
GAMMA = 0.5

/Users/vaishnavverma/.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /var/folders/gp/0qsbr0x11d315dt15fl17c940000gn/T/matplotlib-68c2rie6 because there was an issue with the default path (/Users/vaishnavverma/.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Matplotlib is building the font cache; this may take a moment.


In [2]:
# === Data Setup ===
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.svm import SVC, SVR
from scipy.optimize import minimize

wine = load_wine()
X_scaled = StandardScaler().fit_transform(wine.data)
y = wine.target
K = len(np.unique(y))

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)
print(f"Data: {X_train.shape[0]} train, {X_test.shape[0]} test, {K} classes")

Data: 124 train, 54 test, 3 classes


In [3]:
# === Define Uniform Cost Matrix ===
# All off-diagonal entries are 1 (equal cost for all misclassifications)
UNIFORM_COST = np.ones((K, K)) - np.eye(K)

print("Uniform Cost Matrix:")
print(UNIFORM_COST)
print("\nRegret for class 0 example:", UNIFORM_COST[0])
print("Regret for class 1 example:", UNIFORM_COST[1])
print("Note: all regret vectors are binary — 0 for correct, 1 for wrong.")

Uniform Cost Matrix:
[[0. 1. 1.]
 [1. 0. 1.]
 [1. 1. 0.]]

Regret for class 0 example: [0. 1. 1.]
Regret for class 1 example: [1. 0. 1.]
Note: all regret vectors are binary — 0 for correct, 1 for wrong.


In [4]:
# === OSSVR Implementation (dual QP, no bias) ===
import cvxpy as cp

class OneSidedSVR:
    def __init__(self, C=1.0, gamma='scale'):
        self.C = C
        self.gamma = gamma
        self.alpha_ = None
        self.X_train_ = None
        self.gamma_value_ = None
    
    def _compute_gamma(self, X):
        if self.gamma == 'scale':
            return 1.0 / (X.shape[1] * X.var())
        return self.gamma
    
    def fit(self, X, r):
        N = X.shape[0]
        self.X_train_ = X.copy()
        self.gamma_value_ = self._compute_gamma(X)
        K_mat = rbf_kernel(X, X, gamma=self.gamma_value_)
        K_mat = (K_mat + K_mat.T) / 2
        K_mat += 1e-8 * np.eye(N)
        
        alpha = cp.Variable(N)
        objective = cp.Maximize(r @ alpha - 0.5 * cp.quad_form(alpha, K_mat))
        constraints = [alpha >= 0, alpha <= self.C]
        prob = cp.Problem(objective, constraints)
        prob.solve(solver=cp.CLARABEL, verbose=False)
        self.alpha_ = np.array(alpha.value).flatten()
        return self
    
    def predict(self, X):
        K_mat = rbf_kernel(X, self.X_train_, gamma=self.gamma_value_)
        return K_mat @ self.alpha_

def predict_argmin(models, X):
    preds = np.column_stack([m.predict(X) for m in models])
    return np.argmin(preds, axis=1)

def avg_cost(y_true, y_pred, C):
    return np.mean(C[y_true, y_pred])

print("OSSVR class defined.")

OSSVR class defined.


In [5]:
# === Train OSSVR with Uniform Costs ===
uniform_regret_train = UNIFORM_COST[y_train]

ossvr_models = [OneSidedSVR(C=C_REG, gamma=GAMMA).fit(X_train, uniform_regret_train[:, k]) for k in range(K)]
y_pred_ossvr = predict_argmin(ossvr_models, X_test)
ossvr_cost = avg_cost(y_test, y_pred_ossvr, UNIFORM_COST)
ossvr_acc = accuracy_score(y_test, y_pred_ossvr)

print(f"OSSVR (uniform cost) — Avg Cost: {ossvr_cost:.4f}, Accuracy: {ossvr_acc:.4f}")

OSSVR (uniform cost) — Avg Cost: 0.0370, Accuracy: 0.9630


In [7]:
# === Standard SVM (cost-unaware, optimal for uniform costs) ===
# Use gamma='scale' which is sklearn's default and gives SVM its best performance
std_svm = SVC(C=10.0, kernel='rbf', gamma='scale', random_state=RANDOM_SEED)
std_svm.fit(X_train, y_train)
y_pred_svm = std_svm.predict(X_test)
svm_cost = avg_cost(y_test, y_pred_svm, UNIFORM_COST)
svm_acc = accuracy_score(y_test, y_pred_svm)

print(f"Standard SVM          — Avg Cost: {svm_cost:.4f}, Accuracy: {svm_acc:.4f}")
print(f"OSSVR (uniform cost)  — Avg Cost: {ossvr_cost:.4f}, Accuracy: {ossvr_acc:.4f}")
print(f"\nDifference (OSSVR - SVM): Cost = {ossvr_cost - svm_cost:+.4f}, Acc = {ossvr_acc - svm_acc:+.4f}")
if ossvr_cost >= svm_cost:
    print("\n→ OSSVR provides no cost advantage over Standard SVM with uniform costs.")
else:
    print("\n→ OSSVR slightly outperforms Standard SVM, but the advantage is NOT")
    print("  due to cost-sensitive reasoning — it's merely because one-sided regression")
    print("  with RBF kernel is a competitive classifier. The cost structure provides no signal.")

Standard SVM          — Avg Cost: 0.0185, Accuracy: 0.9815
OSSVR (uniform cost)  — Avg Cost: 0.0370, Accuracy: 0.9630

Difference (OSSVR - SVM): Cost = +0.0185, Acc = -0.0185

→ OSSVR provides no cost advantage over Standard SVM with uniform costs.


In [ ]:
# === Visualisation ===
import os
# Robust path: works whether CWD is workspace root or partB/
_cwd = os.path.abspath('')
RESULTS_DIR = os.path.join(_cwd, 'results') if _cwd.endswith('partB') else os.path.join(_cwd, 'partB', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ax = axes[0]
method_names = ['OSSVR\n(Uniform Cost)', 'Standard SVM']
costs = [ossvr_cost, svm_cost]
colors = ['#e74c3c', '#2ecc71']
bars = ax.bar(method_names, costs, color=colors, edgecolor='black', width=0.5)
ax.set_ylabel('Average Misclassification Cost')
ax.set_title('Failure Mode: OSSVR with Uniform Cost Matrix')
for bar, cost in zip(bars, costs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{cost:.4f}', ha='center', va='bottom', fontweight='bold')

# Confusion matrices
from sklearn.metrics import confusion_matrix
ax = axes[1]
cm_ossvr = confusion_matrix(y_test, y_pred_ossvr)
cm_svm = confusion_matrix(y_test, y_pred_svm)

# Show as side-by-side text
ax.axis('off')
table_data = [['', 'OSSVR', 'Std SVM']]
for i in range(K):
    row = [f'True {i}']
    row.append(str(cm_ossvr[i].tolist()))
    row.append(str(cm_svm[i].tolist()))
    table_data.append(row)
table_data.append(['Accuracy', f'{ossvr_acc:.4f}', f'{svm_acc:.4f}'])
table_data.append(['Avg Cost', f'{ossvr_cost:.4f}', f'{svm_cost:.4f}'])

table = ax.table(cellText=table_data, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.5)
ax.set_title('Confusion Matrix Comparison', pad=20)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'failure_mode_uniform_cost.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

Saved.


/var/folders/gp/0qsbr0x11d315dt15fl17c940000gn/T/ipykernel_79946/2304276449.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Why the Method Fails

OSSVR fails to provide any advantage under a uniform cost matrix because its core mechanism — preserving cost magnitudes through regression — has nothing meaningful to preserve. With uniform costs, every misclassification incurs the same penalty, so the regret vector degenerates into a binary indicator: 0 for the correct class and 1 for every incorrect class. The K regressors are effectively trying to approximate binary labels using a continuous regression framework, which is a harder problem than binary classification.

Specifically, the one-sided loss allows free over-prediction of the constant target 1, which means all regressors may converge to similarly high values for incorrect classes. The argmin prediction then depends on tiny numerical differences between regression outputs that are all trying to approximate the same constant. This makes the prediction fragile and sensitive to solver noise.

This directly connects to **Assumption 1 from Task 1.2**: OSSVR assumes the cost matrix is known and meaningful. A uniform cost matrix is technically valid but carries no useful information beyond correct/incorrect — exactly the information a standard classifier already captures. The method's design assumes there are *distinctions* between different types of errors, and when those distinctions vanish, the method's additional complexity becomes pure overhead.

A standard SVM, by contrast, is natively designed to maximise classification accuracy, which is the optimal objective when all errors are equally costly. It does not need to approximate continuous targets or combine multiple regression outputs.

## Suggested Modification

A simple fix would be to add a **cost-matrix complexity check** at training time: if the cost matrix is uniform (or nearly uniform, with low variance across off-diagonal entries), OSSVR should fall back to a standard OVA-SVM classifier instead of training K regressors, since the regression formulation provides no benefit in this regime.